In [17]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('homework_8.1.csv')
df.drop(columns='Unnamed: 0', inplace=True)
df

,X,Y,Z
0,1,4.109218,1.764052
1,0,2.259504,0.400157
2,0,-0.647584,0.978738
3,0,2.106071,2.240893
4,1,3.583464,1.867558
...,...,...,...
995,1,1.757928,0.412871
996,1,0.631199,-0.198399
997,0,1.104620,0.094192
998,0,-0.651432,-1.147611


### Problem 1-2

Question 1: Using homework_8.1.csv, find the average treatment effect (ATE) with inverse probability weighting. Then, include your code and a written explanation of your work (mentioning any choices or strategies you made in writing the code) in your homework reflection.  

Here are some steps to follow: 

* Estimate the propensity scores using logistic regression. Fit the model so that the Z values predict X. 

* Use the model to predict the propensity scores (e.g., using predict_proba if you are using sklearn). 

* Calculate inverse probability weights (1/P for X=1 and 1/(1-P) for X=0). 

* Estimate the ATE (the Y difference between X=1 and X=0, using the appropriate weights for each). 

Then, the ATE is closest to: 

In [ ]:
from sklearn.linear_model import LogisticRegression


model = LogisticRegression(random_state=42)

# Fit the model so that the Z values predict X
model.fit(df[['Z']], df['X'])

df['Propensity'] = model.predict_proba(df[['Z']])[:, -1]
df

,X,Y,Z,Propensity
0,1,4.109218,1.764052,0.840114
1,0,2.259504,0.400157,0.584646
2,0,-0.647584,0.978738,0.711082
3,0,2.106071,2.240893,0.892793
4,1,3.583464,1.867558,0.853089
...,...,...,...,...
995,1,1.757928,0.412871,0.587624
996,1,0.631199,-0.198399,0.441226
997,0,1.104620,0.094192,0.511594
998,0,-0.651432,-1.147611,0.239959


In [18]:
df['IPW'] = (1 / df['Propensity']).where(df['X'] == 1, np.nan)
df['IPW'] = df['IPW'].mask(df['X'] == 0, 1 / (1 - df['Propensity']))
df

,X,Y,Z,Propensity,IPW
0,1,4.109218,1.764052,0.840114,1.190315
1,0,2.259504,0.400157,0.584646,2.407585
2,0,-0.647584,0.978738,0.711082,3.461195
3,0,2.106071,2.240893,0.892793,9.327719
4,1,3.583464,1.867558,0.853089,1.172211
...,...,...,...,...,...
995,1,1.757928,0.412871,0.587624,1.701767
996,1,0.631199,-0.198399,0.441226,2.266412
997,0,1.104620,0.094192,0.511594,2.047478
998,0,-0.651432,-1.147611,0.239959,1.315719


In [21]:
# Estimate the ATE (the Y difference between X=1 and X=0, using the appropriate weights for each). 
y_treatment_1_weighted = np.sum(df["Y"] * df["X"] * df["IPW"]) / np.sum(
    df["X"] * df["IPW"]
)

y_treatment_0_weighted = np.sum(df["Y"] * (1 - df["X"]) * df["IPW"]) / np.sum(
    (1 - df["X"]) * df["IPW"]
)

In [22]:
len(df[df['X'] == 1])

481

In [23]:
y_treatment_1_weighted - y_treatment_0_weighted

np.float64(2.2743411898510137)

It is closest to D. 2.3

Question 2: The propensity scores of the first three items in the dataframe are closest to: 



In [24]:
df.head(3)

,X,Y,Z,Propensity,IPW
0,1,4.109218,1.764052,0.840114,1.190315
1,0,2.259504,0.400157,0.584646,2.407585
2,0,-0.647584,0.978738,0.711082,3.461195


They are closest to 0.84, 0.58, 0.71 (B)